# Student Exam Performance Analysis & Multi-Model ML Exploration (v2.0)

## 1. Project Background & Analytical Context
Educational institutions need early, explainable indicators to identify students at risk of falling behind and simulate the impact of targeted behavioral interventions before final exams.

In v2.0, this project scales up to **20,000 primary cohort records** with **20 raw features** and **8 leak-free engineered features** (28 total pipeline features), cross-checked alongside real **UCI Student Performance** (`1,044` rows) and **Hugging Face 10K** (`10,000` rows) benchmark datasets and a **9-model ML tournament** (including LightGBM, HistGradientBoosting, MLP Neural Network, and Stacking Ensemble).

In [ ]:
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
from src.features import engineer_features, ENGINEERED_FEATURES

df = pd.read_csv(Path('../data/student_data.csv'))
print(f"Primary Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

### 1.1 Data Hygiene & Missing Value Analysis
Real-world educational telemetry frequently contains incomplete survey entries across sleep, tutoring, screen time, commute duration, and parental surveys. Before modeling, we inspect exact missingness rates across all 20 raw predictors.

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
data_quality_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Percent': missing_pct})
print("=== Missing Value Audit ===")
print(data_quality_df[data_quality_df['Missing_Count'] > 0])
print("\n=== Numeric Feature Summary ===")
df.describe().round(2)

### 1.2 Target Distribution & Variance Characterization
Next, we examine `final_score` across our 20,000-student cohort to verify spread, central tendency, and absence of pathological label collapse.

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.hist(df['final_score'], bins=30, color='#6366f1', edgecolor='#1e293b', alpha=0.8)
plt.axvline(df['final_score'].mean(), color='#e11d48', linestyle='--', linewidth=2, label=f"Mean: {df['final_score'].mean():.1f}")
plt.axvline(df['final_score'].median(), color='#10b981', linestyle=':', linewidth=2, label=f"Median: {df['final_score'].median():.1f}")
plt.title('Distribution of Student Final Exam Scores (n = 20,000)', fontsize=12, fontweight='bold')
plt.xlabel('Final Score (0-100)')
plt.ylabel('Student Count')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

### 1.3 Feature Engineering & Correlation Signals
Using `src.features.engineer_features()`, we derive 8 composite academic and wellbeing features (`study_efficiency`, `academic_momentum`, `sleep_deficit`, `engagement_score`, `study_sleep_balance`, `digital_distraction_ratio`, `formative_delta`, `wellbeing_index`) and measure linear correlations with `final_score`. Inside the scikit-learn pipeline, `AcademicFeatureEngineer` learns medians strictly from `X_train` to guarantee zero data leakage.

In [ ]:
df_eng = engineer_features(df)
numeric_cols = df_eng.select_dtypes(include=[np.number]).columns
target_corr = df_eng[numeric_cols].corr()['final_score'].drop('final_score').sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#e11d48' if v < 0 else '#4f46e5' for v in target_corr.values]
target_corr.plot(kind='barh', color=colors, ax=ax)
ax.set_title('Pearson Correlation of Raw & Engineered Features with Final Exam Score', fontweight='bold')
ax.set_xlabel('Correlation Coefficient (r)')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(True, axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

### 1.4 External Real Datasets & 9-Model Benchmark Summary
Finally, we inspect the cached real Hugging Face / UCI external benchmarks (`data/external/`) and review the persisted 9-model cross-validation tournament results (`reports/benchmark_results.json`).

In [ ]:
uci_path = Path('../data/external/uci_student_performance.csv')
hf10k_path = Path('../data/external/hf_student_performance_10k.csv')
if uci_path.exists():
    print('UCI Student Performance:', pd.read_csv(uci_path).shape)
if hf10k_path.exists():
    print('HF 10K Student Performance:', pd.read_csv(hf10k_path).shape)

bench_path = Path('../reports/benchmark_results.json')
if bench_path.exists():
    report = json.loads(bench_path.read_text(encoding='utf-8'))
    print(f"Champion Model: {report['champion_model']} | Holdout Test R2 = {report['test_metrics']['R2']}")
    display(pd.DataFrame(report['models'])[['name', 'cv_rmse', 'val_rmse', 'test_rmse', 'test_r2', 'train_time_sec']])